In [9]:
# Allow importing from src/ when this notebook runs inside notebooks/
import sys, os
sys.path.insert(0, os.path.abspath(".."))

# Reload modules during development so edits to src/ take effect immediately
%load_ext autoreload
%autoreload 2

from src.config import PROJECT_ROOT, BRFSS_XPT
print("PROJECT_ROOT :", PROJECT_ROOT)
print("BRFSS_XPT    :", BRFSS_XPT, "exists:", BRFSS_XPT.exists())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
PROJECT_ROOT : /Users/ht/Desktop/aly6150_diabetes_project/aly6150-diabetes-health-project
BRFSS_XPT    : /Users/ht/Desktop/aly6150_diabetes_project/aly6150-diabetes-health-project/data/raw/brfss_2023/LLCP2023.XPT exists: True


In [10]:
import pandas as pd
import numpy as np
import pyreadstat

# BRFSS contains non-UTF-8 characters in variable labels — specify latin1 encoding
print("Loading BRFSS — this takes about a minute...")
brfss, brfss_meta = pyreadstat.read_xport(str(BRFSS_XPT), encoding="latin1")

print(f"\nBRFSS shape: {brfss.shape}")
print(f"First 10 columns: {brfss.columns[:10].tolist()}")
print(f"\nDtype summary:")
print(brfss.dtypes.value_counts())

Loading BRFSS — this takes about a minute...

BRFSS shape: (433323, 350)
First 10 columns: ['_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO', '_PSU', 'CTELENM1']

Dtype summary:
float64    345
str          5
Name: count, dtype: int64


In [11]:
# Define a search helper to find columns by keyword

def search_columns(df, keywords):
    """Return columns whose names contain any of the given keywords (case-insensitive)."""
    keywords = [k.upper() for k in keywords]
    return [col for col in df.columns if any(k in col.upper() for k in keywords)]

In [12]:
# BRFSS Variable Discovery — revised for 2023 file
#
# BRFSS variable naming conventions:
#   - Leading underscore = CDC-calculated / derived variable (use these when possible)
#   - Numeric suffix on raw response variables (DIABETE4, CHECKUP1) tracks question version
#   - Missing codes: 7 (Don't know), 9 (Refused), 77/99 for two-digit items, blank for skipped
#
# Known 2023 file structure (confirmed via inspection):
#   - Insurance: use _HLTHPL1 (not HLTHPLN1); PRIMINS1 also available
#   - Race: use _IMPRACE (imputed); _RACEPRV and _RACEGR3 also present
#   - Arthritis: use _DRDXAR2 (not _DRDXAR1)
#   - Foot check (FEETCHK*): NOT in 2023 — dropped from preventive index
#   - Diabetes education (DIABEDU, DOCTDIAB): NOT in 2023 — dropped

print("STATE/GEO   :", search_columns(brfss, ["_STATE", "MSCODE", "_URBSTAT"]))
print("AGE         :", search_columns(brfss, ["_AGEG5YR", "_AGE65YR", "_AGE80"]))
print("SEX         :", search_columns(brfss, ["SEXVAR", "_SEX"]))
print("RACE/ETHN   :", search_columns(brfss, ["_RACE", "_HISPAN", "_IMPRACE"]))
print("INCOME      :", search_columns(brfss, ["INCOME", "_INCOMG"]))
print("EDUCATION   :", search_columns(brfss, ["EDUCA", "_EDUCAG"]))
print("DIABETES    :", search_columns(brfss, ["DIABETE", "DIABAGE",
                                              "INSULIN", "CHKHEMO", "EYEEXAM"]))
print("ACCESS      :", search_columns(brfss, ["MEDCOST", "_HLTHPL", "PERSDOC",
                                              "CHECKUP", "PRIMINS"]))
print("HEALTH      :", search_columns(brfss, ["GENHLTH", "PHYSHLTH", "MENTHLTH",
                                              "POORHLTH", "_RFHLTH"]))
print("BMI/WEIGHT  :", search_columns(brfss, ["_BMI"]))
print("SMOKING     :", search_columns(brfss, ["_SMOKER", "_RFSMOK", "SMOKDAY",
                                              "LASTSMK", "STOPSMK"]))
print("ACTIVITY    :", search_columns(brfss, ["_PA150", "_PAINDX", "_PAREC",
                                              "EXERANY", "_TOTINDA"]))
print("COMORB      :", search_columns(brfss, ["BPHIGH", "BPMEDS", "TOLDHI",
                                              "CVDINFR", "CVDCRHD", "CVDSTRK",
                                              "ASTHMA", "_LTASTH", "_DRDXAR",
                                              "CHCKDNY", "CHCCOPD"]))
print("PREVENTIVE  :", search_columns(brfss, ["_FLSHOT", "_PNEUMO", "FLUSHOT"]))
print("WT/STSTR    :", search_columns(brfss, ["_LLCPWT", "_STSTR", "_PSU"]))

STATE/GEO   : ['_STATE', '_URBSTAT', 'MSCODE']
AGE         : ['_AGEG5YR', '_AGE65YR', '_AGE80']
SEX         : ['SEXVAR', '_SEX']
RACE/ETHN   : ['_IMPRACE', '_HISPANC', '_RACE', '_RACEG21', '_RACEGR3', '_RACEPRV']
INCOME      : ['INCOME3', '_INCOMG1']
EDUCATION   : ['EDUCA', '_EDUCAG']
DIABETES    : ['DIABETE4', 'DIABAGE4', 'INSULIN1', 'CHKHEMO3', 'EYEEXAM1']
ACCESS      : ['PRIMINS1', 'PERSDOC3', 'MEDCOST1', 'CHECKUP1', '_HLTHPL1']
HEALTH      : ['GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', '_RFHLTH']
BMI/WEIGHT  : ['_BMI5', '_BMI5CAT']
SMOKING     : ['SMOKDAY2', 'LASTSMK2', 'STOPSMK2', '_SMOKER3', '_RFSMOK3']
ACTIVITY    : ['EXERANY2', '_TOTINDA', '_PAINDX3', '_PA150R4', '_PAREC3']
COMORB      : ['BPHIGH6', 'BPMEDS1', 'TOLDHI3', 'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'ASTHMA3', 'CHCCOPD3', 'CHCKDNY2', '_LTASTH1', '_DRDXAR2']
PREVENTIVE  : ['FLUSHOT7', '_FLSHOT7', '_PNEUMO3']
WT/STSTR    : ['_PSU', '_STSTR', '_LLCPWT2', '_LLCPWT']


In [13]:
brfss_labels = dict(zip(brfss_meta.column_names, brfss_meta.column_labels))

# ============================================================
# Confirmed available in 2023 BRFSS (LLCP2023):
#   Insurance        : use _HLTHPL1 (HLTHPLN1 was renamed)
#   Race/ethnicity   : use _IMPRACE (imputed; _RACEPRV and _RACEGR3 also present)
#   Arthritis        : use _DRDXAR2 (was _DRDXAR1)
#   Asthma           : ASTHMA3 (current) + _LTASTH1 (lifetime)
#   Physical activity: _PA150R4
#
# Confirmed NOT in 2023:
#   Foot check       : FEETCHK1/2/3 all missing — preventive index drops to 3 items
#   Diabetes ed      : DIABEDU, DOCTDIAB missing — drop
# ============================================================

# Variables grouped by purpose; matches the structure of clean_brfss()
key_vars_grouped = {
    "Sampling design": [
        "_STATE", "_LLCPWT", "_STSTR", "_PSU",
    ],
    "Demographics": [
        "_AGEG5YR", "SEXVAR", "_IMPRACE", "EDUCA",
        "INCOME3", "_INCOMG1",
        "MSCODE", "_URBSTAT",
    ],
    "Diabetes diagnosis + self-management": [
        "DIABETE4", "DIABAGE4", "INSULIN1",
        "CHKHEMO3", "EYEEXAM1",
    ],
    "Access to care": [
        "MEDCOST1", "_HLTHPL1", "PRIMINS1", "PERSDOC3", "CHECKUP1",
    ],
    "Self-rated health": [
        "GENHLTH", "PHYSHLTH", "MENTHLTH", "POORHLTH",
    ],
    "Risk behaviors": [
        "_SMOKER3", "_RFSMOK3", "_BMI5", "_BMI5CAT",
        "_PA150R4", "EXERANY2", "_TOTINDA",
    ],
    "Comorbidities": [
        "BPHIGH6", "BPMEDS1", "TOLDHI3",
        "CVDINFR4", "CVDCRHD4", "CVDSTRK3",
        "ASTHMA3", "_LTASTH1", "CHCKDNY2", "CHCCOPD3", "_DRDXAR2",
    ],
    "Preventive care": [
        "_FLSHOT7", "_PNEUMO3",
    ],
}

# Flatten for the summary counts
key_vars = [v for sublist in key_vars_grouped.values() for v in sublist]


def suggest_alternative(missing_var, all_cols):
    """For a missing variable, suggest similar columns that exist in the file."""
    # Strip leading underscore and trailing digits to find the stem
    stem = missing_var.lstrip("_").rstrip("0123456789")
    if len(stem) < 3:
        return []
    matches = [c for c in all_cols if stem.upper() in c.upper() and c != missing_var]
    return matches[:5]  # cap at 5 suggestions


# ============================================================
# Print label table, grouped
# ============================================================
print(f"{'Variable':<15} {'':2}{'Label':<60}")
print("=" * 80)

for section, cols in key_vars_grouped.items():
    print(f"\n--- {section} ---")
    for col in cols:
        if col in brfss_labels:
            label = brfss_labels[col]
            print(f"  {col:<13} ✓ {label[:60]}")
        else:
            print(f"  {col:<13} ✗ (NOT IN FILE)")

# ============================================================
# Summary + recovery suggestions for any missing variables
# ============================================================
present = [c for c in key_vars if c in brfss_labels]
missing = [c for c in key_vars if c not in brfss_labels]

print("\n" + "=" * 80)
print(f"Variables PRESENT: {len(present)} of {len(key_vars)}")
print(f"Variables MISSING: {len(missing)}")

if missing:
    print("\nRecovery suggestions for missing variables:")
    print("(Similar columns that exist in the file — verify in codebook before substituting)")
    for var in missing:
        alts = suggest_alternative(var, brfss.columns)
        if alts:
            print(f"  {var:<13} → consider: {alts}")
        else:
            print(f"  {var:<13} → no similar columns found; drop from analysis")
else:
    print("\n✓ All key variables confirmed present in 2023 BRFSS file.")

Variable          Label                                                       

--- Sampling design ---
  _STATE        ✓ STATE FIPS CODE
  _LLCPWT       ✓ FINAL WEIGHT: LAND-LINE AND CELL-PHONE D
  _STSTR        ✓ SAMPLE DESIGN STRATIFICATION VARIABLE
  _PSU          ✓ PRIMARY SAMPLING UNIT

--- Demographics ---
  _AGEG5YR      ✓ REPORTED AGE IN FIVE-YEAR AGE CATEGORIES
  SEXVAR        ✓ SEX OF RESPONDENT
  _IMPRACE      ✓ IMPUTED RACE/ETHNICITY VALUE
  EDUCA         ✓ EDUCATION LEVEL
  INCOME3       ✓ INCOME LEVEL
  _INCOMG1      ✓ COMPUTED INCOME CATEGORIES
  MSCODE        ✓ METROPOLITAN STATUS CODE
  _URBSTAT      ✓ URBAN/RURAL STATUS

--- Diabetes diagnosis + self-management ---
  DIABETE4      ✓ (EVER TOLD) YOU HAD DIABETES
  DIABAGE4      ✓ AGE WHEN FIRST TOLD YOU HAD DIABETES
  INSULIN1      ✓ NOW TAKING INSULIN
  CHKHEMO3      ✓ TIMES CHECKED FOR GLYCOSYLATED HEMOGLOBI
  EYEEXAM1      ✓ LAST EYE EXAM WHERE PUPILS WERE DILATED

--- Access to care ---
  MEDCOST1      ✓ COULD NOT

In [14]:
def show_dist(series, name, note=""):
    """Show value distribution for a single series."""
    print(f"\n--- {name} ---")
    if note:
        print(f"  ({note})")
    print(series.value_counts(dropna=False).sort_index().to_string())

print("=" * 60)
print("OUTCOME & PREDICTOR DISTRIBUTIONS (full BRFSS)")
print("=" * 60)

show_dist(brfss["MEDCOST1"], "MEDCOST1",
          "Could not see doctor due to cost; 1=Yes, 2=No, 7=DK, 9=Refused")

show_dist(brfss["_HLTHPL1"], "_HLTHPL1",
          "Have any health insurance; 1=Yes, 2=No, 9=DK/Refused/Missing")

show_dist(brfss["GENHLTH"], "GENHLTH",
          "General health; 1=Excellent ... 5=Poor; 7=DK, 9=Refused")

show_dist(brfss["CHECKUP1"], "CHECKUP1",
          "Last routine checkup; 1=Within past year, ... 8=Never, 7=DK, 9=Refused")

show_dist(brfss["_BMI5CAT"], "_BMI5CAT",
          "BMI category; 1=Under, 2=Normal, 3=Overweight, 4=Obese")

show_dist(brfss["_SMOKER3"], "_SMOKER3",
          "Smoking status; 1=Daily, 2=Some days, 3=Former, 4=Never, 9=DK/Refused")

OUTCOME & PREDICTOR DISTRIBUTIONS (full BRFSS)

--- MEDCOST1 ---
  (Could not see doctor due to cost; 1=Yes, 2=No, 7=DK, 9=Refused)
MEDCOST1
1.0     37198
2.0    394587
7.0      1174
9.0       362
NaN         2

--- _HLTHPL1 ---
  (Have any health insurance; 1=Yes, 2=No, 9=DK/Refused/Missing)
_HLTHPL1
1.0    391946
2.0     22703
9.0     18674

--- GENHLTH ---
  (General health; 1=Excellent ... 5=Poor; 7=DK, 9=Refused)
GENHLTH
1.0     63410
2.0    142115
3.0    144209
4.0     61955
5.0     20372
7.0       897
9.0       361
NaN         4

--- CHECKUP1 ---
  (Last routine checkup; 1=Within past year, ... 8=Never, 7=DK, 9=Refused)
CHECKUP1
1.0    348057
2.0     38031
3.0     20970
4.0     17737
7.0      5168
8.0      2747
9.0       611
NaN         2

--- _BMI5CAT ---
  (BMI category; 1=Under, 2=Normal, 3=Overweight, 4=Obese)
_BMI5CAT
1.0      6767
2.0    116500
3.0    139615
4.0    129906
NaN     40535

--- _SMOKER3 ---
  (Smoking status; 1=Daily, 2=Some days, 3=Former, 4=Never, 9=DK/Refus

In [15]:
print("=" * 60)
print("OUTCOME & PREDICTOR DISTRIBUTIONS (full BRFSS)")
print("=" * 60)

show_dist(brfss["MEDCOST1"], "MEDCOST1",
          "Could not see doctor due to cost; 1=Yes, 2=No, 7=DK, 9=Refused")

show_dist(brfss["_HLTHPL1"], "_HLTHPL1",
          "Have any health insurance; 1=Yes, 2=No, 9=DK/Refused/Missing")

show_dist(brfss["GENHLTH"], "GENHLTH",
          "General health; 1=Excellent ... 5=Poor; 7=DK, 9=Refused")

show_dist(brfss["CHECKUP1"], "CHECKUP1",
          "Last routine checkup; 1=Within past year, ... 8=Never, 7=DK, 9=Refused")

show_dist(brfss["_BMI5CAT"], "_BMI5CAT",
          "BMI category; 1=Under, 2=Normal, 3=Overweight, 4=Obese")

show_dist(brfss["_SMOKER3"], "_SMOKER3",
          "Smoking status; 1=Daily, 2=Some days, 3=Former, 4=Never, 9=DK/Refused")

OUTCOME & PREDICTOR DISTRIBUTIONS (full BRFSS)

--- MEDCOST1 ---
  (Could not see doctor due to cost; 1=Yes, 2=No, 7=DK, 9=Refused)
MEDCOST1
1.0     37198
2.0    394587
7.0      1174
9.0       362
NaN         2

--- _HLTHPL1 ---
  (Have any health insurance; 1=Yes, 2=No, 9=DK/Refused/Missing)
_HLTHPL1
1.0    391946
2.0     22703
9.0     18674

--- GENHLTH ---
  (General health; 1=Excellent ... 5=Poor; 7=DK, 9=Refused)
GENHLTH
1.0     63410
2.0    142115
3.0    144209
4.0     61955
5.0     20372
7.0       897
9.0       361
NaN         4

--- CHECKUP1 ---
  (Last routine checkup; 1=Within past year, ... 8=Never, 7=DK, 9=Refused)
CHECKUP1
1.0    348057
2.0     38031
3.0     20970
4.0     17737
7.0      5168
8.0      2747
9.0       611
NaN         2

--- _BMI5CAT ---
  (BMI category; 1=Under, 2=Normal, 3=Overweight, 4=Obese)
_BMI5CAT
1.0      6767
2.0    116500
3.0    139615
4.0    129906
NaN     40535

--- _SMOKER3 ---
  (Smoking status; 1=Daily, 2=Some days, 3=Former, 4=Never, 9=DK/Refus

In [16]:
print("=" * 60)
print("OUTCOME & PREDICTOR DISTRIBUTIONS")
print("=" * 60)
print("\nFirst: distributions on the FULL BRFSS file (national context).")
print("Then: same distributions on the ANALYTIC SUBSET (all 11 target states + age 45-64 + diabetes).")

# All 11 target state FIPS codes
from src.brfss_cleaning import TARGET_STATES
TARGET_FIPS = list(TARGET_STATES.keys())
AGE_45_64_CODES = [6, 7, 8, 9]

analytic_mask = (
    brfss["_STATE"].isin(TARGET_FIPS)
    & brfss["_AGEG5YR"].isin(AGE_45_64_CODES)
    & (brfss["DIABETE4"] == 1)
)
brfss_analytic = brfss[analytic_mask]

print(f"\nFull sample size:     {len(brfss):>7,}")
print(f"Analytic sample size: {len(brfss_analytic):>7,}")
print(f"\nState breakdown (analytic subset):")
print(brfss_analytic["_STATE"].map(TARGET_STATES).value_counts().to_string())


def show_dual_dist(col, name, note=""):
    """Show distribution on full file and on analytic subset side by side."""
    print(f"\n--- {name} ---")
    if note:
        print(f"  ({note})")
    full = brfss[col].value_counts(dropna=False).sort_index()
    sub  = brfss_analytic[col].value_counts(dropna=False).sort_index()
    combined = pd.DataFrame({
        "full_n":    full,
        "full_%":    (full / full.sum() * 100).round(1),
        "analytic_n": sub.reindex(full.index, fill_value=0),
        "analytic_%": (sub.reindex(full.index, fill_value=0) / sub.sum() * 100).round(1),
    })
    print(combined.to_string())


# OUTCOMES & ACCESS
print("\n" + "=" * 60)
print("OUTCOMES & ACCESS VARIABLES")
print("=" * 60)

show_dual_dist("MEDCOST1", "MEDCOST1",
               "Could not see doctor due to cost; 1=Yes, 2=No, 7=DK, 9=Refused")

show_dual_dist("_HLTHPL1", "_HLTHPL1",
               "Have any health insurance; 1=Yes, 2=No, 9=DK/Refused/Missing")

show_dual_dist("PERSDOC3", "PERSDOC3",
               "Have personal doctor; 1=Yes (one), 2=Yes (more than one), 3=No, 7=DK, 9=Refused")

show_dual_dist("CHECKUP1", "CHECKUP1",
               "Routine checkup: 1=past year, 2=1-2yr, 3=2-5yr, 4=5+yr, 7=DK, 8=Never, 9=Refused")

# DEMOGRAPHICS
print("\n" + "=" * 60)
print("DEMOGRAPHIC VARIABLES")
print("=" * 60)

show_dual_dist("SEXVAR",   "SEXVAR",   "1=Male, 2=Female")
show_dual_dist("_IMPRACE", "_IMPRACE", "1=NH White, 2=NH Black, 3=Asian, 4=AI/AN, 5=Hispanic, 6=Other")
show_dual_dist("EDUCA",    "EDUCA",    "1=No school...4=GED/HS...6=College grad; 9=Refused")
show_dual_dist("INCOME3",  "INCOME3",  "1=<$10K, 2=10-15K, ... 11=200K+; 77=DK, 99=Refused")

# SELF-RATED HEALTH
print("\n" + "=" * 60)
print("SELF-RATED HEALTH")
print("=" * 60)

show_dual_dist("GENHLTH", "GENHLTH",
               "1=Excellent, 2=VeryGood, 3=Good, 4=Fair, 5=Poor; 7=DK, 9=Refused")

print("\n--- Numeric day-count variables (PHYSHLTH, MENTHLTH, POORHLTH) ---")
print("Note: 88=None (zero days, VALID), 77=Don't know, 99=Refused")
for col in ["PHYSHLTH", "MENTHLTH", "POORHLTH"]:
    print(f"\n{col} — analytic sample:")
    counts = brfss_analytic[col].value_counts(dropna=False).sort_index()
    special = counts[counts.index.isin([77, 88, 99]) | counts.index.isna()]
    print(f"  Special codes:\n{special.to_string()}")
    valid = brfss_analytic.loc[brfss_analytic[col].between(1, 30), col]
    print(f"  Valid range (1-30) summary:")
    print(f"    n={len(valid):,}, mean={valid.mean():.1f}, median={valid.median():.1f}")

# RISK BEHAVIORS
print("\n" + "=" * 60)
print("RISK BEHAVIORS")
print("=" * 60)

show_dual_dist("_BMI5CAT", "_BMI5CAT",
               "1=Under, 2=Normal, 3=Overweight, 4=Obese; NaN if missing height/weight")
show_dual_dist("_SMOKER3", "_SMOKER3",
               "1=Daily, 2=Some days, 3=Former, 4=Never, 9=DK/Refused")
show_dual_dist("_PA150R4", "_PA150R4",
               "1=Meets both guidelines, 2=Aerobic only, 3=Neither, 4=Strengthening only, 9=DK")

# COMORBIDITIES
print("\n" + "=" * 60)
print("COMORBIDITY FLAGS (analytic subset only)")
print("=" * 60)
print("All expected: 1=Yes, 2=No, 7=DK, 9=Refused")

comorb_vars = ["BPHIGH6", "TOLDHI3", "CVDINFR4", "CVDCRHD4",
               "CVDSTRK3", "CHCCOPD3", "CHCKDNY2", "_DRDXAR2"]
for col in comorb_vars:
    counts = brfss_analytic[col].value_counts(dropna=False).sort_index()
    print(f"\n{col}:")
    print(counts.to_string())

# SOCIAL DETERMINANTS 
print("\n" + "=" * 60)
print("SOCIAL DETERMINANTS MODULE 29 (analytic subset)")
print("=" * 60)
print("Note: NaN = state did not run Module 29")
sdoh_vars = ["FOODSTMP", "SDHFOOD1", "SDHBILLS", "SDHUTILS", "SDHTRNSP"]
for col in sdoh_vars:
    if col in brfss_analytic.columns:
        n_valid = brfss_analytic[col].notna().sum()
        print(f"\n{col} (n_valid={n_valid:,}):")
        print(brfss_analytic[col].value_counts(dropna=False).sort_index().to_string())
    else:
        print(f"\n{col}: NOT IN FILE")


OUTCOME & PREDICTOR DISTRIBUTIONS

First: distributions on the FULL BRFSS file (national context).
Then: same distributions on the ANALYTIC SUBSET (all 11 target states + age 45-64 + diabetes).

Full sample size:     433,323
Analytic sample size:   3,589

State breakdown (analytic subset):
_STATE
Maine            504
Louisiana        403
Connecticut      399
Massachusetts    335
West Virginia    316
Arkansas         314
Alabama          299
Mississippi      274
Rhode Island     270
Vermont          251
New Hampshire    224

OUTCOMES & ACCESS VARIABLES

--- MEDCOST1 ---
  (Could not see doctor due to cost; 1=Yes, 2=No, 7=DK, 9=Refused)
          full_n  full_%  analytic_n  analytic_%
MEDCOST1                                        
1.0        37198     8.6         412        11.5
2.0       394587    91.1        3167        88.2
7.0         1174     0.3           8         0.2
9.0          362     0.1           2         0.1
NaN            2     0.0           0         0.0

--- _HLTHPL1 

In [17]:
from src.brfss_cleaning import clean_brfss
from src.config import DATA_PROCESSED

print("=" * 60)
print("BRFSS CLEANING")
print("=" * 60)

brfss_clean = clean_brfss(brfss)

BRFSS CLEANING
  Step 2a — after target state filter: 75,170 (removed 358,153)
  Step 2b — after age 45-64 filter:  24,165 (removed 51,005)
  Step 2c — after diabetes filter:    3,589 (removed 20,576)

  Final analytic sample: 3,589 people across 11 states
  Regions: {'New England': 1983, 'Lower-income South': 1606}
  Missed-care-cost rate: 11.5%
  Uninsured rate:        3.5%
  Mean comorbidity count: 2.50
  SDOH-module eligible:  773
  Preventive-care eligible subgroup: 676 (states that ran Module 2)
    Mean preventive-care index (0-2, eligible only): 1.17
  Columns: 51


In [18]:
# Sanity checks

print(f"Shape: {brfss_clean.shape}\n")

print("Region breakdown:")
print(brfss_clean["region"].value_counts().to_string())

print("\nState breakdown:")
print(brfss_clean["state_name"].value_counts().to_string())

print("\nSex:")
print(brfss_clean["sex_label"].value_counts(dropna=False).to_string())

print("\nRace/ethnicity:")
print(brfss_clean["race_ethnicity"].value_counts(dropna=False).to_string())

print("\nAge band (collapsed):")
print(brfss_clean["age_band_collapsed"].value_counts().to_string())

print("\nIncome tier:")
print(brfss_clean["income_tier"].value_counts(dropna=False).to_string())

print("\nKey binary outcomes (mean = rate):")
for col in ["missed_care_cost", "uninsured", "recent_checkup", "obese",
            "current_smoker", "meets_pa_guideline", "poor_health",
            "recent_a1c", "recent_eye_exam", "flu_shot"]:
    n    = brfss_clean[col].notna().sum()
    rate = brfss_clean[col].mean()
    print(f"  {col:<25} rate={rate:.1%}  (n_valid={n:,})")

print("\nComorbidity count distribution:")
print(brfss_clean["comorbidity_count"].value_counts(dropna=False).sort_index().to_string())

print("\nPreventive care index distribution (Module 2 eligible only):")
eligible = brfss_clean[brfss_clean["preventive_care_eligible"] == 1]
print(f"  n_eligible = {len(eligible):,}")
print(eligible["preventive_care_index"].value_counts(dropna=False).sort_index().to_string())

print("\nSDOH module — eligible respondents by region:")
sdoh_eligible = brfss_clean[brfss_clean["sdoh_eligible"] == 1]
print(f"  n_sdoh_eligible = {len(sdoh_eligible):,}")
if len(sdoh_eligible) > 0:
    print(sdoh_eligible["region"].value_counts().to_string())
    print("\n  SDOH burden index distribution (eligible only):")
    print(sdoh_eligible["sdoh_burden_index"].value_counts(dropna=False).sort_index().to_string())
else:
    print("  No SDOH-eligible respondents found — Module 29 may not have run in target states.")
    print("  (This will be confirmed when the actual data file is loaded.)")


Shape: (3589, 51)

Region breakdown:
region
New England           1983
Lower-income South    1606

State breakdown:
state_name
Maine            504
Louisiana        403
Connecticut      399
Massachusetts    335
West Virginia    316
Arkansas         314
Alabama          299
Mississippi      274
Rhode Island     270
Vermont          251
New Hampshire    224

Sex:
sex_label
Female    1809
Male      1780

Race/ethnicity:
race_ethnicity
NH White          2576
NH Black           555
Hispanic           236
Other/Multiple     121
NH Asian            52
AI/AN               49

Age band (collapsed):
age_band_collapsed
55-64    2406
45-54    1183

Income tier:
income_tier
<$25K      787
$25-50K    777
$50-75K    752
$75K+      704
NaN        569

Key binary outcomes (mean = rate):
  missed_care_cost          rate=11.5%  (n_valid=3,579)
  uninsured                 rate=3.5%  (n_valid=3,457)
  recent_checkup            rate=94.6%  (n_valid=3,564)
  obese                     rate=62.7%  (n_valid=3,3

In [19]:
# Save to folder 
from src.config import DATA_PROCESSED

output_path = DATA_PROCESSED / "brfss_diabetes_2023_clean.csv"
brfss_clean.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")

Saved: /Users/ht/Desktop/aly6150_diabetes_project/aly6150-diabetes-health-project/data/processed/brfss_diabetes_2023_clean.csv
File size: 853.3 KB


In [20]:
# Confirm it loads back correctly
import pandas as pd
brfss_reload = pd.read_csv(output_path)
print(f"Reloaded shape: {brfss_reload.shape}")
print(f"Columns match: {list(brfss_reload.columns) == list(brfss_clean.columns)}")

Reloaded shape: (3589, 51)
Columns match: True
